# Create Transportation Asset JSON

Generates a MacroEnergy.jl transportation asset JSON from a cost-per-ton-per-km and a dictionary of province-pair distances.

In [ ]:
import json
import os

## Inputs

In [ ]:
# Commodity to transport (used in vertex names, IDs, and the edge commodity field)
commodity = "CrudeSteel"

# Lowercase prefix used in vertex names and IDs, e.g. "crudesteel" -> "crudesteel_Region1Beijing"
commodity_prefix = commodity.lower()

# Cost in $/ton/km
cost_per_ton_per_km = 0.1

# Dictionary: keys are (start_province, end_province) tuples, values are distances in km.
# Province names must match the region labels used in vertex names (e.g. "Region1Beijing").
# Each pair produces one directed (unidirectional) transport link.
province_pair_distances = {
    ("Region1Beijing",  "Region2Tianjin"):        39,
    ("Region1Beijing",  "Region3Hebei"):           90,
    ("Region2Tianjin",  "Region3Hebei"):           80,
    # Add more pairs here ...
}

# Output path
output_path = (
    "/Users/al3792/Documents_Local/MacroEnergy.jl/"
    "ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288/"
    f"assets/assets_1/{commodity_prefix}_transport.json"
)

## Build and write JSON

In [ ]:
def make_transport_json(
    commodity: str,
    commodity_prefix: str,
    cost_per_ton_per_km: float,
    province_pair_distances: dict,
) -> dict:
    """Return the transport asset dict ready for json.dump."""
    instances = []
    for (origin, destination), distance_km in province_pair_distances.items():
        variable_om_cost = round(cost_per_ton_per_km * distance_km, 4)
        instances.append({
            "id": f"{commodity_prefix}_transport_{origin}_to_{destination}",
            "edges": {
                "transmission_edge": {
                    "start_vertex": f"{commodity_prefix}_{origin}",
                    "end_vertex":   f"{commodity_prefix}_{destination}",
                    "variable_om_cost": variable_om_cost,
                }
            },
        })

    return {
        "transmission": {
            "global_data": {
                "transforms": {},
                "edges": {
                    "transmission_edge": {
                        "unidirectional": True,
                        "has_capacity": False,
                        "commodity": commodity,
                        "loss_fraction": 0,
                    }
                },
            },
            "instance_data": instances,
        }
    }


data = make_transport_json(
    commodity=commodity,
    commodity_prefix=commodity_prefix,
    cost_per_ton_per_km=cost_per_ton_per_km,
    province_pair_distances=province_pair_distances,
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Wrote {len(province_pair_distances)} transport links to:")
print(output_path)

## Preview first few entries

In [ ]:
preview = {
    "transmission": {
        "global_data": data["transmission"]["global_data"],
        "instance_data": data["transmission"]["instance_data"][:5],
    }
}
print(json.dumps(preview, indent=2))